# 04 — Parse Model Predictions

Questo notebook trasforma gli output grezzi prodotti dal notebook 03
nelle predizioni utilizzabili per il calcolo delle metriche.

Il parser adotta un approccio conservativo:

- accetta direttamente le label restituite nel formato previsto;
- recupera risposte semplici come `The label is compliant`;
- rifiuta risposte contenenti più label o interpretazioni ambigue;
- accetta per STS-B esclusivamente punteggi interi da 0 a 5;
- mantiene inalterate le risposte del task generativo;
- non elimina gli output invalidi, che saranno conteggiati nel notebook 05.

Il parsing non utilizza la ground truth per interpretare la risposta.

In [1]:
from pathlib import Path
from collections import Counter
import json
import re
import unicodedata

import pandas as pd


PROJECT_ROOT = Path(
    r"C:\Users\fede2\Desktop\Project"
).resolve()

INPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "predictions"
    / "hf_predictions_raw.jsonl"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "predictions"
    / "hf_predictions_parsed.jsonl"
)

INVALID_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "predictions"
    / "hf_predictions_invalid.jsonl"
)

SUMMARY_PATH = (
    PROJECT_ROOT
    / "data"
    / "predictions"
    / "parsing_summary.csv"
)

PARSER_VERSION = "1.0"

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"File delle predizioni grezze non trovato:\n{INPUT_PATH}\n\n"
        "Eseguire prima il notebook 03."
    )

print("Input:", INPUT_PATH)
print("Output:", OUTPUT_PATH)

Input: C:\Users\fede2\Desktop\Project\data\predictions\hf_predictions_raw.jsonl
Output: C:\Users\fede2\Desktop\Project\data\predictions\hf_predictions_parsed.jsonl


In [2]:
TASK_LABELS = {
    "sst2_style": [
        "compliant",
        "non_compliant",
    ],

    "paraphrase_style": [
        "equivalent",
        "not_equivalent",
    ],

    "stsb_style": [
        0,
        1,
        2,
        3,
        4,
        5,
    ],

    "qqp_style": [
        "duplicate",
        "not_duplicate",
    ],

    "mnli_style": [
        "entailment",
        "contradiction",
        "neutral",
    ],

    "rte_style": [
        "entailment",
        "not_entailment",
    ],

    "bias_detection": [
        "biased",
        "unbiased",
    ],

    "grounded_answer_generation": None,
}


FORBIDDEN_TASKS = {
    "qnli_style",
}


# Varianti linguistiche accettate soltanto quando
# identificano in modo chiaro una label.
TASK_ALIASES = {
    "sst2_style": {
        "compliant": [],
        "non_compliant": [
            "noncompliant",
        ],
    },

    "paraphrase_style": {
        "equivalent": [],
        "not_equivalent": [
            "not equivalent",
        ],
    },

    "qqp_style": {
        "duplicate": [],
        "not_duplicate": [
            "not duplicate",
        ],
    },

    "mnli_style": {
        "entailment": [],
        "contradiction": [],
        "neutral": [],
    },

    "rte_style": {
        "entailment": [
            "entailed",
        ],
        "not_entailment": [
            "not entailed",
        ],
    },

    "bias_detection": {
        "biased": [],
        "unbiased": [],
    },
}


print("Task configurati:", len(TASK_LABELS))

Task configurati: 8


In [3]:
def read_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

            except json.JSONDecodeError as error:
                raise ValueError(
                    f"JSON non valido alla riga {line_number}: "
                    f"{error}"
                ) from error

            if not isinstance(record, dict):
                raise TypeError(
                    f"La riga {line_number} non contiene "
                    "un oggetto JSON."
                )

            records.append(record)

    return records


def write_jsonl(
    path: Path,
    records: list[dict],
) -> None:
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

In [4]:
raw_predictions = read_jsonl(
    INPUT_PATH
)

required_fields = {
    "example_id",
    "task_name",
    "model_name",
    "y_true",
    "raw_output",
    "status",
}

errors = []

for index, record in enumerate(
    raw_predictions,
    start=1,
):
    missing_fields = (
        required_fields
        - set(record)
    )

    if missing_fields:
        errors.append(
            f"Record {index}: campi mancanti "
            f"{sorted(missing_fields)}"
        )

    task_name = record.get(
        "task_name"
    )

    if task_name in FORBIDDEN_TASKS:
        errors.append(
            f"Record {index}: task escluso {task_name!r}"
        )

    elif task_name not in TASK_LABELS:
        errors.append(
            f"Record {index}: task sconosciuto {task_name!r}"
        )


prediction_keys = [
    (
        record.get("model_name"),
        record.get("example_id"),
    )
    for record in raw_predictions
]

duplicate_keys = [
    key
    for key, count
    in Counter(prediction_keys).items()
    if count > 1
]

if duplicate_keys:
    errors.append(
        "Coppie duplicate (model_name, example_id): "
        + ", ".join(
            str(key)
            for key in duplicate_keys[:20]
        )
    )


if errors:
    raise RuntimeError(
        "Predizioni grezze non valide:\n- "
        + "\n- ".join(errors)
    )


print("Predizioni caricate:", len(raw_predictions))
print(
    "Modelli:",
    Counter(
        record["model_name"]
        for record in raw_predictions
    ),
)

Predizioni caricate: 800
Modelli: Counter({'base': 160, 'regulation': 160, 'positive_legends': 160, 'negative_legends': 160, 'negative_to_correction': 160})


In [5]:
def normalize_text(value) -> str:
    if value is None:
        return ""

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    text = text.strip()

    # Rimuove eventuali code fence esterne.
    text = re.sub(
        r"^\s*```(?:\w+)?\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\s*```\s*$",
        "",
        text,
    )

    text = text.lower()

    text = (
        text
        .replace("–", "-")
        .replace("—", "-")
        .replace("’", "'")
    )

    # Per il confronto delle label:
    # non_compliant, non-compliant e non compliant
    # diventano la stessa forma.
    text = re.sub(
        r"[_-]+",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


def strip_outer_punctuation(text: str) -> str:
    return text.strip(
        " \t\n\r"
        ".,;:!?"
        "\"'`"
        "()[]{}"
        "*#"
    )

In [6]:
def build_normalized_aliases(
    task_name: str,
) -> dict[str, str]:
    """
    Restituisce una mappa:

        variante normalizzata -> label canonica
    """

    aliases = {}

    for canonical_label in TASK_LABELS[task_name]:
        canonical_text = str(
            canonical_label
        )

        variants = [
            canonical_text,
            *TASK_ALIASES.get(
                task_name,
                {},
            ).get(
                canonical_label,
                [],
            ),
        ]

        for variant in variants:
            normalized_variant = normalize_text(
                variant
            )

            aliases[
                normalized_variant
            ] = canonical_label

    return aliases


NORMALIZED_ALIASES = {
    task_name: build_normalized_aliases(
        task_name
    )
    for task_name, labels in TASK_LABELS.items()
    if labels is not None
    and task_name != "stsb_style"
}


for task_name, aliases in NORMALIZED_ALIASES.items():
    print(
        task_name,
        "->",
        aliases,
    )

sst2_style -> {'compliant': 'compliant', 'non compliant': 'non_compliant', 'noncompliant': 'non_compliant'}
paraphrase_style -> {'equivalent': 'equivalent', 'not equivalent': 'not_equivalent'}
qqp_style -> {'duplicate': 'duplicate', 'not duplicate': 'not_duplicate'}
mnli_style -> {'entailment': 'entailment', 'contradiction': 'contradiction', 'neutral': 'neutral'}
rte_style -> {'entailment': 'entailment', 'entailed': 'entailment', 'not entailment': 'not_entailment', 'not entailed': 'not_entailment'}
bias_detection -> {'biased': 'biased', 'unbiased': 'unbiased'}


In [7]:
ANSWER_PREFIX_PATTERN = re.compile(
    r"^(?:the\s+)?"
    r"(?:answer|label|prediction|classification|class|result|assistant)"
    r"\s*(?:is|=|:)?\s+"
    r"(.+?)"
    r"[.!?]?$",
    flags=re.IGNORECASE,
)


def alias_pattern(alias: str) -> re.Pattern:
    words = [
        re.escape(word)
        for word in alias.split()
    ]

    phrase = r"\s+".join(
        words
    )

    return re.compile(
        rf"(?<![a-z0-9]){phrase}(?![a-z0-9])",
        flags=re.IGNORECASE,
    )


def find_label_matches(
    normalized_output: str,
    task_name: str,
) -> list[dict]:
    matches = []

    aliases = NORMALIZED_ALIASES[
        task_name
    ]

    for alias, canonical_label in aliases.items():
        pattern = alias_pattern(
            alias
        )

        for match in pattern.finditer(
            normalized_output
        ):
            matches.append({
                "canonical_label": canonical_label,
                "alias": alias,
                "start": match.start(),
                "end": match.end(),
            })

    # Ordina prima le espressioni più lunghe.
    # In questo modo "not entailed" prevale su "entailed".
    matches.sort(
        key=lambda item: (
            -(item["end"] - item["start"]),
            item["start"],
        )
    )

    filtered_matches = []

    for candidate in matches:
        contained = any(
            candidate["start"] >= existing["start"]
            and candidate["end"] <= existing["end"]
            and (
                candidate["start"] > existing["start"]
                or candidate["end"] < existing["end"]
            )
            for existing in filtered_matches
        )

        if not contained:
            filtered_matches.append(
                candidate
            )

    return filtered_matches


def parse_classification_output(
    raw_output,
    task_name: str,
) -> dict:
    normalized_output = normalize_text(
        raw_output
    )

    cleaned_output = strip_outer_punctuation(
        normalized_output
    )

    if not cleaned_output:
        return {
            "y_pred": None,
            "is_valid_prediction": False,
            "parse_status": "invalid_empty_output",
            "parse_method": None,
            "matched_labels": [],
            "normalized_output": normalized_output,
        }

    aliases = NORMALIZED_ALIASES[
        task_name
    ]

    # Caso ideale: il modello restituisce soltanto la label.
    if cleaned_output in aliases:
        canonical_label = aliases[
            cleaned_output
        ]

        canonical_normalized = normalize_text(
            canonical_label
        )

        method = (
            "exact_label"
            if cleaned_output == canonical_normalized
            else "exact_alias"
        )

        return {
            "y_pred": canonical_label,
            "is_valid_prediction": True,
            "parse_status": "valid",
            "parse_method": method,
            "matched_labels": [
                canonical_label
            ],
            "normalized_output": normalized_output,
        }

    # Recupera formati semplici come:
    # "The label is compliant."
    prefix_match = ANSWER_PREFIX_PATTERN.fullmatch(
        cleaned_output
    )

    if prefix_match:
        candidate = strip_outer_punctuation(
            normalize_text(
                prefix_match.group(1)
            )
        )

        if candidate in aliases:
            canonical_label = aliases[
                candidate
            ]

            return {
                "y_pred": canonical_label,
                "is_valid_prediction": True,
                "parse_status": "valid",
                "parse_method": "answer_wrapper",
                "matched_labels": [
                    canonical_label
                ],
                "normalized_output": normalized_output,
            }

    # Cerca label chiaramente presenti nel testo.
    label_matches = find_label_matches(
        normalized_output,
        task_name,
    )

    matched_labels = sorted({
        match["canonical_label"]
        for match in label_matches
    })

    if len(matched_labels) == 1:
        return {
            "y_pred": matched_labels[0],
            "is_valid_prediction": True,
            "parse_status": "valid",
            "parse_method": "unique_embedded_label",
            "matched_labels": matched_labels,
            "normalized_output": normalized_output,
        }

    if len(matched_labels) > 1:
        return {
            "y_pred": None,
            "is_valid_prediction": False,
            "parse_status": "invalid_ambiguous_labels",
            "parse_method": None,
            "matched_labels": matched_labels,
            "normalized_output": normalized_output,
        }

    return {
        "y_pred": None,
        "is_valid_prediction": False,
        "parse_status": "invalid_unrecognized_output",
        "parse_method": None,
        "matched_labels": [],
        "normalized_output": normalized_output,
    }

In [8]:
SCORE_PREFIX_PATTERN = re.compile(
    r"^(?:the\s+)?"
    r"(?:score|answer|rating|prediction|result)"
    r"\s*(?:is|=|:)?\s*"
    r"([0-5])"
    r"[.!?]?$",
    flags=re.IGNORECASE,
)


def parse_stsb_output(
    raw_output,
) -> dict:
    normalized_output = normalize_text(
        raw_output
    )

    cleaned_output = strip_outer_punctuation(
        normalized_output
    )

    if not cleaned_output:
        return {
            "y_pred": None,
            "is_valid_prediction": False,
            "parse_status": "invalid_empty_output",
            "parse_method": None,
            "matched_scores": [],
            "normalized_output": normalized_output,
        }

    # Output ideale: "4"
    if re.fullmatch(
        r"[0-5]",
        cleaned_output,
    ):
        return {
            "y_pred": int(cleaned_output),
            "is_valid_prediction": True,
            "parse_status": "valid",
            "parse_method": "exact_integer",
            "matched_scores": [
                int(cleaned_output)
            ],
            "normalized_output": normalized_output,
        }

    # Output recuperabile: "Score: 4"
    prefix_match = SCORE_PREFIX_PATTERN.fullmatch(
        cleaned_output
    )

    if prefix_match:
        score = int(
            prefix_match.group(1)
        )

        return {
            "y_pred": score,
            "is_valid_prediction": True,
            "parse_status": "valid",
            "parse_method": "score_wrapper",
            "matched_scores": [
                score
            ],
            "normalized_output": normalized_output,
        }

    # I decimali non sono ammessi.
    decimal_numbers = re.findall(
        r"(?<!\d)[+-]?\d+\.\d+(?!\d)",
        cleaned_output,
    )

    if decimal_numbers:
        return {
            "y_pred": None,
            "is_valid_prediction": False,
            "parse_status": "invalid_decimal_score",
            "parse_method": None,
            "matched_scores": decimal_numbers,
            "normalized_output": normalized_output,
        }

    # Cerca tutti gli interi autonomi presenti nell'output.
    integer_tokens = re.findall(
        r"(?<![\d.])[+-]?\d+(?![\d.])",
        cleaned_output,
    )

    integer_values = [
        int(value)
        for value in integer_tokens
    ]

    unique_values = sorted(
        set(integer_values)
    )

    if len(unique_values) == 1:
        score = unique_values[0]

        if score in {
            0,
            1,
            2,
            3,
            4,
            5,
        }:
            return {
                "y_pred": score,
                "is_valid_prediction": True,
                "parse_status": "valid",
                "parse_method": "unique_embedded_integer",
                "matched_scores": unique_values,
                "normalized_output": normalized_output,
            }

        return {
            "y_pred": None,
            "is_valid_prediction": False,
            "parse_status": "invalid_out_of_range_score",
            "parse_method": None,
            "matched_scores": unique_values,
            "normalized_output": normalized_output,
        }

    if len(unique_values) > 1:
        return {
            "y_pred": None,
            "is_valid_prediction": False,
            "parse_status": "invalid_ambiguous_scores",
            "parse_method": None,
            "matched_scores": unique_values,
            "normalized_output": normalized_output,
        }

    return {
        "y_pred": None,
        "is_valid_prediction": False,
        "parse_status": "invalid_unrecognized_output",
        "parse_method": None,
        "matched_scores": [],
        "normalized_output": normalized_output,
    }

In [9]:
def parse_prediction_record(
    record: dict,
) -> dict:
    task_name = record[
        "task_name"
    ]

    raw_output = record.get(
        "raw_output"
    )

    inference_status = record.get(
        "status"
    )

    # Gli errori del notebook 03 non vengono interpretati.
    if inference_status != "completed":
        parsing_result = {
            "y_pred": None,
            "is_valid_prediction": False,
            "parse_status": "upstream_inference_error",
            "parse_method": None,
            "normalized_output": "",
        }

    elif task_name == "grounded_answer_generation":
        generated_answer = (
            str(raw_output).strip()
            if raw_output is not None
            else ""
        )

        if generated_answer:
            parsing_result = {
                "y_pred": generated_answer,
                "is_valid_prediction": True,
                "parse_status": "generation_passthrough",
                "parse_method": "passthrough",
                "normalized_output": generated_answer,
            }

        else:
            parsing_result = {
                "y_pred": None,
                "is_valid_prediction": False,
                "parse_status": "invalid_empty_generation",
                "parse_method": None,
                "normalized_output": "",
            }

    elif task_name == "stsb_style":
        parsing_result = parse_stsb_output(
            raw_output
        )

    else:
        parsing_result = parse_classification_output(
            raw_output,
            task_name,
        )

    return {
        **record,
        **parsing_result,
        "parser_version": PARSER_VERSION,
    }


parsed_predictions = [
    parse_prediction_record(record)
    for record in raw_predictions
]

print(
    "Predizioni analizzate:",
    len(parsed_predictions),
)

Predizioni analizzate: 800


In [10]:
validation_errors = []

if len(parsed_predictions) != len(raw_predictions):
    validation_errors.append(
        "Il numero di record analizzati non coincide "
        "con il numero di input."
    )


for record in parsed_predictions:
    task_name = record[
        "task_name"
    ]

    example_id = record[
        "example_id"
    ]

    model_name = record[
        "model_name"
    ]

    y_pred = record[
        "y_pred"
    ]

    is_valid = record[
        "is_valid_prediction"
    ]

    if not is_valid:
        continue

    if task_name == "grounded_answer_generation":
        expected_output = str(
            record["raw_output"]
        ).strip()

        if y_pred != expected_output:
            validation_errors.append(
                f"{model_name}/{example_id}: "
                "risposta generativa modificata."
            )

    elif task_name == "stsb_style":
        if (
            not isinstance(y_pred, int)
            or isinstance(y_pred, bool)
            or y_pred not in TASK_LABELS[task_name]
        ):
            validation_errors.append(
                f"{model_name}/{example_id}: "
                f"punteggio STS-B non valido {y_pred!r}."
            )

    elif y_pred not in TASK_LABELS[task_name]:
        validation_errors.append(
            f"{model_name}/{example_id}: "
            f"label non valida {y_pred!r}."
        )


if validation_errors:
    raise RuntimeError(
        "Validazione del parsing fallita:\n- "
        + "\n- ".join(
            validation_errors[:100]
        )
    )


print("Validazione superata.")

Validazione superata.


In [11]:
summary_df = pd.DataFrame(
    parsed_predictions
)

summary_table = (
    summary_df
    .groupby(
        [
            "model_name",
            "task_name",
        ],
        dropna=False,
    )
    .agg(
        total_predictions=(
            "example_id",
            "count",
        ),
        valid_predictions=(
            "is_valid_prediction",
            "sum",
        ),
    )
    .reset_index()
)

summary_table[
    "invalid_predictions"
] = (
    summary_table["total_predictions"]
    - summary_table["valid_predictions"]
)

summary_table[
    "valid_percentage"
] = (
    summary_table["valid_predictions"]
    / summary_table["total_predictions"]
    * 100
).round(2)

display(summary_table)

,model_name,task_name,total_predictions,valid_predictions,invalid_predictions,valid_percentage
0,base,bias_detection,20,20,0,100.0
1,base,grounded_answer_generation,20,20,0,100.0
2,base,mnli_style,20,20,0,100.0
3,base,paraphrase_style,20,20,0,100.0
4,base,qqp_style,20,20,0,100.0
5,base,rte_style,20,20,0,100.0
6,base,sst2_style,20,20,0,100.0
7,base,stsb_style,20,20,0,100.0
8,negative_legends,bias_detection,20,20,0,100.0
9,negative_legends,grounded_answer_generation,20,20,0,100.0


In [12]:
status_table = (
    summary_df
    .groupby(
        [
            "model_name",
            "task_name",
            "parse_status",
            "parse_method",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        [
            "model_name",
            "task_name",
            "count",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
)

display(status_table)

,model_name,task_name,parse_status,parse_method,count
0,base,bias_detection,valid,exact_label,20
1,base,grounded_answer_generation,generation_passthrough,passthrough,20
2,base,mnli_style,valid,exact_label,20
3,base,paraphrase_style,valid,exact_label,20
4,base,qqp_style,valid,exact_label,20
5,base,rte_style,valid,exact_label,20
6,base,sst2_style,valid,exact_label,20
7,base,stsb_style,valid,exact_integer,20
8,negative_legends,bias_detection,valid,exact_label,20
9,negative_legends,grounded_answer_generation,generation_passthrough,passthrough,20


In [13]:
invalid_predictions = [
    record
    for record in parsed_predictions
    if not record[
        "is_valid_prediction"
    ]
]

print(
    "Output invalidi:",
    len(invalid_predictions),
)

for record in invalid_predictions[:30]:
    print("\n" + "=" * 90)
    print(
        "MODEL:",
        record["model_name"],
    )
    print(
        "TASK:",
        record["task_name"],
    )
    print(
        "ID:",
        record["example_id"],
    )
    print(
        "STATUS:",
        record["parse_status"],
    )
    print(
        "RAW OUTPUT:",
        repr(
            record.get("raw_output")
        ),
    )

Output invalidi: 0


In [14]:
write_jsonl(
    OUTPUT_PATH,
    parsed_predictions,
)

write_jsonl(
    INVALID_OUTPUT_PATH,
    invalid_predictions,
)

summary_table.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)

print("Predizioni complete:")
print(OUTPUT_PATH)

print("\nPredizioni invalide:")
print(INVALID_OUTPUT_PATH)

print("\nRiepilogo:")
print(SUMMARY_PATH)

Predizioni complete:
C:\Users\fede2\Desktop\Project\data\predictions\hf_predictions_parsed.jsonl

Predizioni invalide:
C:\Users\fede2\Desktop\Project\data\predictions\hf_predictions_invalid.jsonl

Riepilogo:
C:\Users\fede2\Desktop\Project\data\predictions\parsing_summary.csv


In [15]:
saved_predictions = read_jsonl(
    OUTPUT_PATH
)

saved_invalid = read_jsonl(
    INVALID_OUTPUT_PATH
)

if len(saved_predictions) != len(raw_predictions):
    raise RuntimeError(
        "Il file finale non contiene tutte le predizioni."
    )

valid_count = sum(
    record["is_valid_prediction"]
    for record in saved_predictions
)

invalid_count = (
    len(saved_predictions)
    - valid_count
)

print("\n" + "=" * 80)
print("NOTEBOOK 04 COMPLETATO")
print("=" * 80)

print("Predizioni totali:", len(saved_predictions))
print("Predizioni valide:", valid_count)
print("Predizioni invalide:", invalid_count)

print("\nDistribuzione parse_status:")

for status, count in Counter(
    record["parse_status"]
    for record in saved_predictions
).most_common():
    print(
        f"- {status}: {count}"
    )

print("\nFile per il notebook 05:")
print(OUTPUT_PATH)


NOTEBOOK 04 COMPLETATO
Predizioni totali: 800
Predizioni valide: 800
Predizioni invalide: 0

Distribuzione parse_status:
- valid: 700
- generation_passthrough: 100

File per il notebook 05:
C:\Users\fede2\Desktop\Project\data\predictions\hf_predictions_parsed.jsonl
